# Open Source, Local, Token Cost

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import ollama
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm import tqdm

## Connect to OpenRouter

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

if openrouter_api_key:
    print('OpenRouter API key loaded successfully,')
else:
    print('Failed to load OpenRouter API key')

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, 
                    api_key=openrouter_api_key,
                    timeout=120,
                    max_retries=5)

OpenRouter API key loaded successfully,


## Part 1: The Data And A Smaller Prompt

In [3]:
DATA = Path("../../../data")

def load(name):
    """ Read one of the JSON seed files"""
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def get_sample_data(tiny=False,
                    held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id. 
    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """

    claims = load('evals/held_out_claims.json' if held_out else 'claims.json')
    decisions = load('evals/held_out_decisions.json' if held_out else 'evals/decisions.json')
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels

In [4]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
claims = [claim for claim in claims if labels[claim['claim_id']]['difficulty'] != 'manual']
policies = {policy['policy_id']: policy for policy in load('policies.json')}
print(f"{len(claims)} claims")

20 claims


In [5]:
decisions = pd.DataFrame([labels[claim['claim_id']] for claim in claims])
decisions['decision'].value_counts()

decision
approve    12
decline     5
refer       3
Name: count, dtype: int64

### A Smaller Prompt

In [6]:
policy_wording= (DATA / 'policy_wording.md').read_text(encoding='utf-8')
short_wording = (DATA / 'policy_wording_short.md').read_text(encoding='utf-8')

print(f'The policy wording is {len(policy_wording.split()):,} words')
print(f'The short policy wording is {len(short_wording.split()):,} words')

The policy wording is 1,369 words
The short policy wording is 808 words


In [7]:
def split_sections(wording):
    """ Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    sections = {part.split("\n")[0].strip(): "##" + part for part in parts[1:]}
    return parts[0], sections

opening, sections = split_sections(short_wording)
list(sections)

['Limits',
 'Cancellation and Interruption',
 'Trip Delay',
 'Baggage Delay',
 'Baggage',
 'Passport',
 'Emergency Medical',
 'Add-ons',
 'Never covered',
 'Payment',
 'Claims']

In [8]:
BENEFITS = {
    'trip_cancellation': 'Cancellation and Interruption',
    'trip_interruption': 'Cancellation and Interruption',
    'trip_delay': 'Trip Delay',
    'baggage_delay': 'Baggage Delay',
    'baggage': 'Baggage',
    'passport': 'Passport',
    'emergency_medical': 'Emergency Medical' 
}

def make_wording(claim):
    """ The sections of the short wording that this claim needs: the limits, 
    its benefits, the add-ons if the policy has any, and the rules for  every claim.
    """
    headings = ['Limits', BENEFITS[claim['claim_type']]]
    if policies[claim['policy_id']]['add_ons']:
        headings.append('Add-ons')
    headings += ['Never covered', 'Payment', 'Claims']
    return '\n\n'.join([opening.strip()] + [sections[heading].strip()for heading in headings])

In [9]:
INSTRUCTIONS = """ You are a claims handler at UnderCover Travel Insurance.
Decide the claim uou are given, using the policy wording below and the customer's 
policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the documents disagree, or the wording does not settle the question.
"""

def make_user_prompt(claim):
    """ The policy record and the claim as compact JSON, without the ids and the text of the documents."""
    policy = policies[claim['policy_id']]
    record = {key: value for key, value in policy.items() if key not in ['policy_id', 'customer_id', 'sold_through', 'status']}
    details = {key: value for key, value in claim.items() if key not in ['policy_id', 'customer_id', 'status']}
    details['documents'] = [{'name': document['name'], 'type': document['type']} for document in claim['documents']]
    return f""" # Policy record
    
{json.dumps(record)}

# Claim

{json.dumps(details)}
"""

def make_messages(claim):
    """ Build the messages for one claim: the task and the the wording it needs,
    then the policy record and the claim.
    """
    return [
        {
            'role': 'system',
            'content': f"{INSTRUCTIONS}\n{make_wording(claim)}\n"
        },
        {
            'role': 'user',
            'content': make_user_prompt(claim)
        },
    ]

In [10]:
messages = make_messages(claims[0])
print(messages[0]['content'])
print(messages[1]['content'])

 You are a claims handler at UnderCover Travel Insurance.
Decide the claim uou are given, using the policy wording below and the customer's 
policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the documents disagree, or the wording does not settle the question.

# UnderCover Travel Insurance: Policy Wording, short version

All amounts are in US dollars. The policy record shows the plan, the add-ons, the trip dates, the trip cost and the travelers.

##Limits

| Benefit | Economy | Premium | First |
|---|---|---|---|
| Cancellation and Interruption maximum | 1,500 | 5,000 | 10,000 |
| Trip Delay waiting period, hours | 12 | 6 | 3 |
| Trip Delay maximum | 300 | 800 | 1,500 |
| Baggage Delay waiting period, hours | 24 | 12 | 6 |
| Baggage Delay ma

## Part 2: Six Open-Weight Models on OpenRouter

In [11]:
class ClaimDecision(BaseModel):
    """ A class handler's decision on one class."""

    rationale: str= Field(description='2-3 sentences to explain the decision along with the relevant section of the policy')
    decision: Literal['approve', 
                      'decline',
                      'refer']
    payout: float = Field(description='The amount to pay in US dollars, or 0 for a decline or a referral')

### Calling the LLM

In [12]:
def decide(claim, model, effort, ignore=None):
    """ Ask one model on OpenRouter to decide one claim, skipping any providers
    listed in ignore.
    """
    start = time.time()
    provider = {'require_parameters': True}

    if ignore:
        provider['ignore'] = ignore
    response = openrouter.chat.completions.parse(
        model = model,
        messages = make_messages(claim),
        response_format = ClaimDecision,
        reasoning_effort = effort,
        extra_body = {'provider': provider}
    )
    answer = response.choices[0].message.parsed
    decision, payout, rationale = answer.decision, answer.payout, answer.rationale
    return{
        'model': model,
        'effort': effort,
        'where': 'OpenRouter',
        'claim_id': claim['claim_id'],
        'decision': decision,
        'payout': payout,
        'rationale': rationale,
        'provider': response.provider,
        'input_tokens': response.usage.prompt_tokens,
        'cost': response.usage.cost,
        'seconds': round(time.time() - start, 1)
    }

In [13]:
first = decide(claims[0], 
               'deepseek/deepseek-v4.1-flash',
               'low')
first

{'model': 'deepseek/deepseek-v4.1-flash',
 'effort': 'low',
 'where': 'OpenRouter',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': 'Trip Delay is covered under the Premium plan: the carrier canceled the return flight due to a crew problem, and the rebooking was the next morning, meeting the 6-hour waiting period. The required carrier confirmation and receipts are attached. The claimed items are meals, accommodation, and local transport during the delay. No deductible applies to Trip Delay, no carrier/other reimbursement is shown, and the total is below the Premium Trip Delay maximum of $800.',
 'provider': 'Makora',
 'input_tokens': 1055,
 'cost': 0.00229965,
 'seconds': 10.1}

### What The Smaller Prompt Saves

In [14]:
def make_day2_messages(claim):
    """ Day 2's prompt: the whole policy wording, and the 
    policy record and the claims as indented JSON
    """
    policy = policies[claim['policy_id']]
    return [
        {
            'role': 'system',
            'content': f"{INSTRUCTIONS}\n{[policy_wording]}\n"
        },
        {
            'role': 'user',
            'content': f"# Policy record\n\n{json.dumps(policy, indent=2)}\n\n# Claim\n\n{json.dumps(claim, indent=2)}\n"
        },
    ]
day2 = openrouter.chat.completions.parse(
    model='deepseek/deepseek-v4.1-flash',
    messages=make_day2_messages(claims[0]),
    response_format=ClaimDecision,
    reasoning_effort='low',
    extra_body={'provider': {'require_parameters': True}},
)

print(f"Day 2's prompt: {day2.usage.prompt_tokens:,} input tokens")
print(f"Today's prompt: {first['input_tokens']:,} input tokens")

Day 2's prompt: 2,808 input tokens
Today's prompt: 1,055 input tokens


### Now Run All 6 Models

In [15]:
def is_right(answer, label):
    """ True when the decision matches the handler's and the payout is within a dollar."""
    return answer['decision'] == label['decision'] and abs(answer['payout'] - label['payout']) <= 1

def score(answer, label):
    """ Add the difficulty, and whether the decision and the payout match the handler's. """
    return {
        **answer,
        'difficulty': label['difficulty'],
        'decision_right': answer['decision'] == label['decision'],
        'right': is_right(answer, label),
    }

def report(results):
    """ Print how a run did: the decisions right, the decisions right with the payout too, the cost and the time. """
    decisions_right = sum(result['decision_right'] for result in results)
    right = sum(result['right'] for result in results)
    cost = sum(result['cost'] for result in results)
    seconds = sum(result['seconds'] for result in results)
    print(f"{results[0]['model']}: {decisions_right} of {len(results)} decisions right, {right} with the payout too, ${cost:.4f} in {seconds:.0f} seconds ")
    

In [16]:
def run(model, effort, claims, labels, ignore=None):
    """ Run a model on OpenRouter on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide(claim, model, effort, ignore)
        results.append(score(answer, labels[claim['claim_id']]))
    report(results)
    return results

In [17]:
mimo = run('xiaomi/mimo-v2.6-pro', None, claims, labels)

xiaomi/mimo-v2.6-pro:   0%|          | 0/20 [00:00<?, ?it/s]

xiaomi/mimo-v2.6-pro: 100%|██████████| 20/20 [06:19<00:00, 18.96s/it]

xiaomi/mimo-v2.6-pro: 16 of 20 decisions right, 14 with the payout too, $0.0205 in 379 seconds 


In [18]:
glm = run('z-ai/glm-5.3-flash', 'low', claims, labels, ignore=['wafer'])

z-ai/glm-5.3-flash: 100%|██████████| 20/20 [02:12<00:00,  6.61s/it]

z-ai/glm-5.3-flash: 16 of 20 decisions right, 13 with the payout too, $0.0043 in 132 seconds 


In [19]:
kimi = run('moonshotai/kimi-k3', 'low', claims, labels)

moonshotai/kimi-k3: 100%|██████████| 20/20 [02:05<00:00,  6.27s/it]

moonshotai/kimi-k3: 16 of 20 decisions right, 15 with the payout too, $0.1445 in 126 seconds 


In [20]:
deepseek = run('deepseek/deepseek-v4.1-flash', 'low', claims, labels)

deepseek/deepseek-v4.1-flash: 100%|██████████| 20/20 [02:01<00:00,  6.08s/it]

deepseek/deepseek-v4.1-flash: 17 of 20 decisions right, 16 with the payout too, $0.0246 in 122 seconds 


In [21]:
nemotron = run('nvidia/nemotron-3-ultra-550b-a55b', 'medium', claims, labels)

nvidia/nemotron-3-ultra-550b-a55b: 100%|██████████| 20/20 [00:36<00:00,  1.82s/it]

nvidia/nemotron-3-ultra-550b-a55b: 16 of 20 decisions right, 13 with the payout too, $0.0219 in 36 seconds 


In [22]:
mistral = run('mistralai/mistral-medium-3-5', 'none', claims, labels)

mistralai/mistral-medium-3-5: 100%|██████████| 20/20 [00:30<00:00,  1.54s/it]

mistralai/mistral-medium-3-5: 15 of 20 decisions right, 12 with the payout too, $0.0531 in 31 seconds 


## Part 3: Three Models On Your Computer With Ollama

In [23]:
print(ollama.pull('qwen3.5:4b').status)
print(ollama.pull('gemma4:e4b').status)
print(ollama.pull('gemma4:e4b').status)

success
success
success


In [24]:
def decide_locally(claim, model):
    """ Ask a model on this computer to decide one claim, through Ollama."""
    start = time.time()
    response = ollama.chat(
        model=model,
        messages=make_messages(claim),
        format=ClaimDecision.model_json_schema(),
        think=False,
        options={'num_ctx': 8192},
    )
    answer = ClaimDecision.model_validate_json(response.message.content)
    return{
        'model': model,
        'effort': 'none',
        'where': 'This computer',
        'claim_id': claim['claim_id'],
        'decision': answer.decision,
        'payout': answer.payout,
        'rationale': answer.rationale,
        'input_tokens': response.prompt_eval_count,
        'cost': 0.0,
        'seconds': round(time.time() - start, 1)
    }

In [25]:
def run_locally(model, claims, labels):
    """ Run a model on this computer on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide_locally(claim, model)
        results.append(score(answer, labels[claim['claim_id']]))
    ollama.generate(model=model, 
                    keep_alive=0)# unload the model, so the next one has the memory.
    report(results)
    return results

In [26]:
first_local = decide_locally(claims[0], 'qwen3.5:4b')
first_local

{'model': 'qwen3.5:4b',
 'effort': 'none',
 'where': 'This computer',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': "The claim involves a flight delay of 24 hours (canceled June 18, rebooked next morning). The policy record shows the 'premium' plan. For the premium plan, the Trip Delay waiting period is 6 hours and the maximum is $1,500. The incident occurred on the return date, which falls within the trip dates. The reason (crew problem) is covered by the policy wording ('mechanical or crew problem'). The claimant provided receipts and a carrier report as required. There are no disclaimers such as traveling while impaired or outside trip dates. Therefore, the claim is approved.",
 'input_tokens': 1138,
 'cost': 0.0,
 'seconds': 7.1}

In [ ]:
# qwen = run_locally('qwen3.5:4b', claims, labels)

In [ ]:
#gemma_e2b = run_locally('gemma_e2b', claims, labels)

In [ ]:
#gemma_e4b = run_locally('gemma_e4b', claims, labels)

In [29]:
cerebras_api_key = os.getenv('CEREBRAS_API_KEY')
CEREBRAS_BASE_URL = 'https://api.cerebras.ai/v1'

if cerebras_api_key:
    print('Cerebras API key found')
    cerebras = OpenAI(base_url=CEREBRAS_BASE_URL,
                      api_key=cerebras_api_key,
                      timeout=120, 
                      max_retries=5)
else:
    print('No Cerebras API key found') 

No Cerebras API key found


In [ ]:
CEREBRAS_INPUT_PRICE = 0.99 / 1_000_000
CEREBRAS_OUTPUT_PRICE = 1.49 / 1_000_000

def decide_on_cerebras(claim, model, effort):
    """ Ask a model on Cerebras to decide one claim."""
    start = time.time()
    response = cerebras.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
    )
    answer =response.choices[0].message.parsed
    usage = response.usage
    return {
        'model': model,
        'effort': effort,
        'where': 'Cerebras',
        'claim_id': claim['claim_id'],
        'decision': answer.decision,
        'payout': 
    }